# AdaTTT VS Code Hybrid Notebook

Hybrid workflow: switch the VS Code kernel between your **local Python environment** (CPU) and the **Colab GPU runtime** as you go. Both kernels share the same project folder via Google Drive sync, so checkpoints, results, and figures are visible to either side.

Each code cell starts with a banner comment so you know which kernel to run it on:

```
# === LOCAL CPU CELL ===   # run on your Mac's Python kernel
# === COLAB GPU CELL ===   # run on the Colab GPU kernel
# === ANY KERNEL ==========# safe in either kernel (setup / sanity)
```

There are no `assert IN_COLAB` guards — the banner is yours to read.


## 0. Runtime Setup (any kernel)

Set `MAX_SAMPLES_VAL = None` for the full 214K validation set. `MAX_SAMPLES_TRAIN = 50000` keeps gate-label generation fast — the gate is a 50K-param MLP and does not need all 443K train samples.

Hybrid split summary:
- **Local CPU**: setup, tests, data prep (annotations only), gate-label generation, analysis, figures, Gradio demo
- **Colab GPU**: COCO image download, base training, feature precompute, all TTT sweeps, gate training, gate sweep, ablations, latency


In [2]:
# === ANY KERNEL ===
import os
import sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/AdaTTT'
    RUNTIME_LABEL = 'colab-gpu'
else:
    # VS Code may start the kernel inside notebooks/; walk up if so.
    cwd = Path.cwd().resolve()
    PROJECT_DIR = str(cwd.parent if cwd.name == 'notebooks' else cwd)
    RUNTIME_LABEL = 'local-cpu'

# Sample budgets (None = full split)
MAX_SAMPLES_VAL = None      # full val for final results
MAX_SAMPLES_TRAIN = 50000   # gate labels only need a subset

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Runtime:', RUNTIME_LABEL)
print('Project directory:', PROJECT_DIR)
print('MAX_SAMPLES_VAL:', MAX_SAMPLES_VAL, '| MAX_SAMPLES_TRAIN:', MAX_SAMPLES_TRAIN)
print('Files:', sorted(os.listdir(PROJECT_DIR))[:20])


Runtime: local-cpu
Project directory: /Users/yugesh/Library/CloudStorage/GoogleDrive-yugeshreddysappidi@gmail.com/My Drive/AdaTTT
MAX_SAMPLES_VAL: None | MAX_SAMPLES_TRAIN: 50000
Files: ['.DS_Store', '.claude', '.git', '.gitignore', '.pytest_cache', 'ADA_TTT_PROJECT_BLUEPRINT.md', 'CLAUDE.md', 'CS518_AdaTTT_Proposal.pdf', 'README.md', 'adattt.egg-info', 'checkpoints', 'config', 'data', 'demo', 'figures', 'gpu', 'logs', 'notebooks', 'pytest.ini', 'requirements.txt']


In [3]:
# === ANY KERNEL ===
# Install the package and dependencies in whichever kernel is active.
!pip install -q -e .
!pip install -q -r requirements.txt

# nvidia-smi only makes sense on Colab GPU.
if IN_COLAB:
    !nvidia-smi
else:
    print('Local kernel — skipping nvidia-smi.')


Local kernel — skipping nvidia-smi.


In [4]:
# === ANY KERNEL ===
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if IN_COLAB and torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif IN_COLAB:
    print('Colab kernel is active but CUDA is not available — change runtime type.')
else:
    print('Local CPU/local Python environment is active.')

from ttt import FullVQAModel, TTTAdapter, AdaptiveRouter
print('AdaTTT imports: OK')


PyTorch: 2.10.0
CUDA available: False
Local CPU/local Python environment is active.
AdaTTT imports: OK


## 1. Local CPU: Tests & Annotation Prep

Run pytest and download VQA-v2 **questions/annotations only** (small JSONs). Image downloads happen on the Colab side in section 1b — that avoids piping 19 GB through your home internet and Google Drive.


In [ ]:
# === LOCAL CPU CELL ===
# pip install was already done in cell 3.
!python -m pytest tests/ -q
!python scripts/01_prepare_data.py --config config/config.yaml --skip-images


============================= test session starts ==============================
platform darwin -- Python 3.12.2, pytest-7.4.4, pluggy-1.0.0
rootdir: /Users/yugesh/Library/CloudStorage/GoogleDrive-yugeshreddysappidi@gmail.com/My Drive/AdaTTT
configfile: pytest.ini
plugins: anyio-3.7.1, dash-3.0.2
collected 94 items                                                             

tests/test_cached_features.py .......                                    [  7%]
tests/test_data_memotion2.py ...........                                 [ 19%]
tests/test_fallback.py .........                                         [ 28%]
tests/test_gate.py ......                                                [ 35%]
tests/test_integration.py ........                                       [ 43%]
tests/test_latency.py ......                                             [ 50%]
tests/test_metrics.py ................                                   [ 67%]
tests/test_models.py .................                      

## 1b. Colab GPU: COCO Image Setup

VQA-v2 image downloads (~19 GB) go directly into the Colab runtime, **not** through your local machine. Strategy:

1. **One-time**: cache the COCO zip files in Drive (`data/coco_zips/`). They're large but only have to be downloaded once for the project's lifetime.
2. **Per session**: extract the zips to `/content/data/` (Colab's local SSD, fast), then symlink `data/train2014` and `data/val2014` to point at the SSD. The `VQADataset` keeps using `data_dir="data/"` from the config — no code changes needed.

Reading 250K small JPEGs from a Drive mount is ~50× slower than from Colab's SSD, which is why we extract locally each session.


In [14]:
# === COLAB GPU CELL ===
# One-time: download COCO zips into Drive so they survive runtime resets.
import os
os.makedirs('data/coco_zips', exist_ok=True)

if not os.path.exists('data/coco_zips/train2014.zip'):
    !wget -c http://images.cocodataset.org/zips/train2014.zip -P data/coco_zips
else:
    print('train2014.zip already cached in Drive')

if not os.path.exists('data/coco_zips/val2014.zip'):
    !wget -c http://images.cocodataset.org/zips/val2014.zip -P data/coco_zips
else:
    print('val2014.zip already cached in Drive')

# Sanity: rough size check
!du -sh data/coco_zips/*.zip 2>/dev/null


train2014.zip already cached in Drive
val2014.zip already cached in Drive
13G	data/coco_zips/train2014.zip
6.2G	data/coco_zips/val2014.zip


In [4]:
# === COLAB GPU CELL ===
# Per session: extract the cached zips to /content/data (Colab SSD), then
# symlink the project's data/ folder so VQADataset finds images via data_dir="data/".
import os, subprocess

LOCAL_DATA = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

for split in ('train2014', 'val2014'):
    target = f'{LOCAL_DATA}/{split}'
    if not os.path.exists(target):
        zip_path = f'data/coco_zips/{split}.zip'
        print(f'Extracting {zip_path} -> {target} ...')
        subprocess.run(['unzip', '-q', zip_path, '-d', LOCAL_DATA], check=True)
    else:
        print(f'{target} already extracted')

    # Symlink data/<split> -> /content/data/<split>
    link = f'data/{split}'
    if os.path.islink(link):
        os.remove(link)
    elif os.path.exists(link):
        raise RuntimeError(f'{link} exists and is not a symlink — remove it manually')
    os.symlink(target, link)
    print(f'Symlinked data/{split} -> {target}')

!ls -la data/train2014 data/val2014 | head -5


Extracting data/coco_zips/train2014.zip -> /content/data/train2014 ...
Symlinked data/train2014 -> /content/data/train2014
Extracting data/coco_zips/val2014.zip -> /content/data/val2014 ...
Symlinked data/val2014 -> /content/data/val2014
lrw------- 1 root root 23 Apr 28 08:44 data/train2014 -> /content/data/train2014
lrw------- 1 root root 21 Apr 28 08:46 data/val2014 -> /content/data/val2014


In [5]:
# === ANY KERNEL ===
# Sanity-check that everything the training scripts need is reachable.
# On local CPU you should see annotations + vocab present and image dirs *missing*.
# On Colab after section 1b you should see all paths present.
from pathlib import Path

required_paths = [
    'data/answer_vocab.json',
    'data/v2_OpenEnded_mscoco_train2014_questions.json',
    'data/v2_mscoco_train2014_annotations.json',
    'data/v2_OpenEnded_mscoco_val2014_questions.json',
    'data/v2_mscoco_val2014_annotations.json',
]
image_paths = ['data/train2014', 'data/val2014']

missing = [p for p in required_paths if not Path(p).exists()]
img_missing = [p for p in image_paths if not Path(p).exists()]

print('Annotation files:')
for p in required_paths:
    print(f'  {"OK" if Path(p).exists() else "MISSING"}  {p}')
print('Image directories:')
for p in image_paths:
    print(f'  {"OK" if Path(p).exists() else "MISSING"}  {p}')

if missing:
    raise FileNotFoundError('Missing annotations: run section 1 first.')
if img_missing and IN_COLAB:
    raise FileNotFoundError('Missing image dirs: run section 1b first.')
print('\nReady.' if not img_missing else '\nLocal kernel: image dirs missing as expected (Colab will set them up).')


Annotation files:
  OK  data/answer_vocab.json
  OK  data/v2_OpenEnded_mscoco_train2014_questions.json
  OK  data/v2_mscoco_train2014_annotations.json
  OK  data/v2_OpenEnded_mscoco_val2014_questions.json
  OK  data/v2_mscoco_val2014_annotations.json
Image directories:
  OK  data/train2014
  OK  data/val2014

Ready.


## 2. Colab GPU: Train Base Model

Trains the fusion + gate auxiliary head + prediction head on top of frozen ViT-B/16 and BERT-base. Best checkpoint goes to `checkpoints/base/best.pt`.


In [ ]:
# === COLAB GPU CELL ===
!python gpu/train_base.py \
    --config config/config.yaml \
    --epochs 15


[06:41:05] INFO - Device: cuda, AMP: True
[06:41:05] INFO - Dataset: vqa_v2
[06:41:05] INFO - Epochs: 15, Batch size: 64, LR: 0.0001
Loading weights: 100% 198/198 [00:00<00:00, 900.62it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 1029.69it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | St

In [22]:
!rm -rf /content/data/val2014
!unzip -q data/coco_zips/val2014.zip -d /content/data/
!ls /content/data/val2014/ | wc -l


40504


## 2b. Colab GPU: Precompute Frozen Encoder Features

Run ViT-B/16 + BERT-base over the dataset **once** and cache the resulting tokens. Every downstream cell that uses `--features` reads from these cached files instead of re-encoding — typically a 3–5× speedup across the entire weekend.

Cached features are stored on Drive (small compared to images: ~few GB total in float16).


In [6]:
# === COLAB GPU CELL ===
import os
os.makedirs('data/features', exist_ok=True)

# Validation features (used by all val sweeps + gate sweep + ablations).
if not os.path.exists('data/features/val.pt'):
    !python gpu/precompute_features.py \
        --config config/config.yaml \
        --split val \
        --output data/features/val.pt \
        --dtype float16
else:
    print('data/features/val.pt already exists — skipping.')

# Train features (only enough for gate labels; subsample with MAX_SAMPLES_TRAIN).
train_max = '' if MAX_SAMPLES_TRAIN is None else f'--max-samples {MAX_SAMPLES_TRAIN}'
if not os.path.exists('data/features/train.pt'):
    !python gpu/precompute_features.py \
        --config config/config.yaml \
        --split train \
        --output data/features/train.pt \
        --dtype float16 \
        {train_max}
else:
    print('data/features/train.pt already exists — skipping.')

!du -sh data/features/*.pt


[02:25:41] INFO - Precompute features: dataset=vqa_v2, split=val
[02:25:42] INFO - Device: cuda, AMP: True, storage dtype: float16
[02:25:42] INFO - Output: data/features/val.pt
config.json: 69.7kB [00:00, 69.1MB/s]
model.safetensors: 100% 346M/346M [00:02<00:00, 139MB/s] 
Loading weights: 100% 198/198 [00:00<00:00, 1203.26it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
config.json: 100% 570/570 [00:00<00:00, 2.55MB/s]
model.safetensors: 100% 440M/

## 3. Colab GPU: TTT Validation Sweeps

Generate validation predictions for the Pareto curve. Every sweep uses cached features (`--features data/features/val.pt`), so the only real work is the K TTT gradient steps per sample.


In [6]:
# === COLAB GPU CELL ===
val_max_arg = '' if MAX_SAMPLES_VAL is None else f'--max-samples {MAX_SAMPLES_VAL}'
train_max_arg = '' if MAX_SAMPLES_TRAIN is None else f'--max-samples {MAX_SAMPLES_TRAIN}'
print('val_max_arg :', val_max_arg or '(full val)')
print('train_max_arg:', train_max_arg or '(full train)')


val_max_arg : (full val)
train_max_arg: --max-samples 50000


In [7]:
# === COLAB GPU CELL ===
# Baseline: no TTT on validation split (fast — just a forward pass).
!python gpu/run_ttt_sweep.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --features data/features/val.pt \
    --split val \
    --k 0 \
    {val_max_arg}


[03:57:35] INFO - TTT Sweep: K=0, objective=masked_patch, dataset=vqa_v2
[03:57:36] INFO - Device: cuda, AMP: True
[03:57:36] INFO - Split: val, encode_batch_size: 64
config.json: 69.7kB [00:00, 77.3MB/s]
model.safetensors: 100% 346M/346M [00:02<00:00, 151MB/s] 
Loading weights: 100% 198/198 [00:00<00:00, 1024.65it/s, Materializing param=layernorm.weight]                                
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
config.json: 100% 570/570 [00:00<00:00, 2.96MB/s]
model.safetensors: 100% 440M/440M [00:02<

In [8]:
# === COLAB GPU CELL ===
# Primary TTT config used by the rest of the pipeline.
!python gpu/run_ttt_sweep.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --features data/features/val.pt \
    --split val \
    --k 1 \
    --objective masked_patch \
    {val_max_arg}


[04:27:49] INFO - TTT Sweep: K=1, objective=masked_patch, dataset=vqa_v2
[04:27:49] INFO - Device: cuda, AMP: True
[04:27:49] INFO - Split: val, encode_batch_size: 64
Loading weights: 100% 198/198 [00:00<00:00, 735.51it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 583.34it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key          

In [9]:
# === COLAB GPU CELL ===
# Optional: additional K and rotation objective for the K-sweep figure.
for k, objective in [(1, 'rotation'), (3, 'masked_patch'), (5, 'masked_patch')]:
    !python gpu/run_ttt_sweep.py \
        --config config/config.yaml \
        --checkpoint checkpoints/base/best.pt \
        --features data/features/val.pt \
        --split val \
        --k {k} \
        --objective {objective} \
        {val_max_arg}


[05:59:50] INFO - TTT Sweep: K=1, objective=rotation, dataset=vqa_v2
[05:59:50] INFO - Device: cuda, AMP: True
[05:59:50] INFO - Split: val, encode_batch_size: 64
Loading weights: 100% 198/198 [00:00<00:00, 698.04it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 678.36it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key              

## 4. Train-Split Predictions for Gate Labels

These two GPU cells produce base + TTT predictions on the (subsampled) train split. The local cell that follows compares them to build gate labels.

`MAX_SAMPLES_TRAIN` (default 50K) keeps this tractable — the gate is a tiny MLP that does not need 443K labeled samples.


In [10]:
# === COLAB GPU CELL ===
!python gpu/run_ttt_sweep.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --features data/features/train.pt \
    --split train \
    --k 0 \
    --resume \
    {train_max_arg}


[17:00:57] INFO - TTT Sweep: K=0, objective=masked_patch, dataset=vqa_v2
[17:00:57] INFO - Device: cuda, AMP: True
[17:00:57] INFO - Split: train, encode_batch_size: 64
Loading weights: 100% 198/198 [00:00<00:00, 732.58it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 606.86it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key        

In [11]:
# === COLAB GPU CELL ===
!python gpu/run_ttt_sweep.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --features data/features/train.pt \
    --split train \
    --k 1 \
    --objective masked_patch \
    --resume \
    {train_max_arg}


[17:05:01] INFO - TTT Sweep: K=1, objective=masked_patch, dataset=vqa_v2
[17:05:01] INFO - Device: cuda, AMP: True
[17:05:01] INFO - Split: train, encode_batch_size: 64
Loading weights: 100% 198/198 [00:00<00:00, 605.45it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 597.08it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key        

In [12]:
# === LOCAL CPU CELL ===
# Pure JSON diff — base vs TTT predictions per sample. No GPU needed.
!python scripts/04_generate_gate_labels.py --config config/config.yaml --split train


Generating Gate Training Labels
  Base predictions: results/ttt_predictions/train/k0_baseline.json
  TTT predictions:  results/ttt_predictions/train/k1_masked_patch.json
  Output:           data/gate_labels_train.json


Results (50000 samples):
  Base correct (label=1.0):   46117  (92.2%)
  TTT helps (label=0.0):        487  (1.0%)
  TTT hurts (label=1.0):       1495  (3.0%)
  Neither (dropped):           1901  (3.8%)

Coverage:
  Total evaluated: 50000
  Kept: 48099 (96.2%)
  Dropped (neither): 1901
  SKIP (1.0): 47612, ADAPT (0.0): 487
  SKIP fraction: 99.0%

✅ Gate labels saved to: data/gate_labels_train.json


## 5. Colab GPU: Train Confidence Gate

Trains only `θ_g` using `data/gate_labels_train.json`. Forward pass goes through the (frozen) encoders + fusion to compute `z`; on Colab GPU this is brief (~30–45 min for 5 epochs).

`train_gate.py` does not currently accept `--features`, so it re-encodes from images. This is one of the few places we still pay the encoder cost — acceptable since gate training is short.


In [13]:
# === COLAB GPU CELL ===
!python gpu/train_gate.py \
    --config config/config.yaml \
    --base-checkpoint checkpoints/base/best.pt \
    --split train


[17:26:31] INFO - Device: cuda
[17:26:31] INFO - Gate training: 5 epochs, batch=128, lr=0.0005
Loading weights: 100% 198/198 [00:00<00:00, 989.76it/s, Materializing param=layernorm.weight]                                  
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 1186.53it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
---------------------

## 6. Colab GPU: Gate Threshold Sweep (Pareto Frontier)

This is the **main deliverable**. `run_gate_sweep.py` runs TTT on all val samples once, saves base + TTT logits, then applies every threshold in `gate_threshold_sweep` from the config post-hoc — one pass instead of five separate `run_inference.py` calls.


In [14]:
# === COLAB GPU CELL ===
!python gpu/run_gate_sweep.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --gate-checkpoint checkpoints/gate/best.pt \
    --features data/features/val.pt \
    --k 1 \
    --objective masked_patch \
    --split val \
    {val_max_arg}


[17:37:03] INFO - Gate sweep: K=1, masked_patch, thresholds=[0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
[17:37:03] INFO - Device: cuda, AMP: True
Loading weights: 100% 198/198 [00:00<00:00, 1079.14it/s, Materializing param=layernorm.weight]                                
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 938.03it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                 

## 7. Colab GPU: Ablation And Latency

The three completed ablations stay as-is. The only remaining ablation run is `ttt_both`, which should be resumed instead of restarted.

Because `ttt_both` uses consistency + mixup, it needs raw COCO `val2014` images in `data/val2014/` even if other val experiments used cached features.


In [ ]:
# === COLAB GPU CELL ===
# Ensure COCO val2014 images exist for the remaining ttt_both ablation.
import os

val_dir = 'data/val2014'
if not os.path.isdir(val_dir) or len(os.listdir(val_dir)) < 100:
    if os.path.exists(val_dir) and not os.path.isdir(val_dir):
        os.remove(val_dir)

    print('Downloading COCO val2014 images (~6 GB)...')
    !wget -q --show-progress http://images.cocodataset.org/zips/val2014.zip -O data/val2014.zip
    !cd data && unzip -q val2014.zip && rm val2014.zip
    print(f'OK: {len(os.listdir(val_dir))} images available in {val_dir}')
else:
    print(f'OK: COCO val2014 already exists with {len(os.listdir(val_dir))} images')


In [6]:
# === COLAB GPU CELL ===
# Resume the remaining ablation without touching completed results.
!python gpu/run_ablation.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --features data/features/val.pt \
    --k 1 \
    --mode ttt_both \
    --resume


[08:46:08] INFO - Ablation: mode=ttt_both, K=1, masked_patch
[08:46:08] INFO -   consistency=True, mixup=True
[08:46:08] INFO - Device: cuda, AMP: True
[08:46:08] INFO - Split: val, encode_batch_size: 64
config.json: 69.7kB [00:00, 66.2MB/s]
model.safetensors: 100% 346M/346M [00:01<00:00, 184MB/s] 
Loading weights: 100% 198/198 [00:00<00:00, 1362.62it/s, Materializing param=layernorm.weight]                                ore.bias]
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
config.json: 100% 570/570 [00:00<00:00, 3.37M

In [17]:
# === COLAB GPU CELL ===
# Latency profiling deliberately does NOT use cached features —
# end-to-end timing must include the encoder forward pass.
!python gpu/run_latency_profile.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --k 1


[11:53:26] INFO - Latency profile: K=1, masked_patch, n=100
[11:53:26] INFO - Device: cuda
Loading weights: 100% 198/198 [00:00<00:00, 1406.17it/s, Materializing param=layernorm.weight]                                
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 1142.85it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
---------------------------

## 8. Local CPU: Analyze Results And Generate Figures

Pure pandas/matplotlib over the JSONs that synced down from Colab via Drive. No GPU.


In [18]:
# === LOCAL CPU CELL ===
!python scripts/02_analyze_results.py --config config/config.yaml
!python scripts/03_generate_figures.py --config config/config.yaml


  Efficient TTT — Results Analysis

📊 Base Model (no TTT)
   Accuracy: 71.65% (95% CI: [69.93%, 73.26%])
   sentiment: 71.65%

📊 TTT: K=0, unknown
   Accuracy: 95.22% (95% CI: [95.03%, 95.41%])
   number: 91.31%
   other: 99.12%
   yes/no: 91.40%
   Est. GFLOPs/sample: 46.3

📊 TTT: K=1, masked_patch
   Accuracy: 93.21% (95% CI: [92.97%, 93.42%])
   number: 86.45%
   other: 98.30%
   yes/no: 88.79%
   Est. GFLOPs/sample: 64.9

📊 TTT: K=0, unknown
   Accuracy: 49.56% (95% CI: [49.34%, 49.76%])
   number: 32.74%
   other: 43.12%
   yes/no: 63.88%
   Est. GFLOPs/sample: 46.3

📊 TTT: K=1, masked_patch
   Accuracy: 48.49% (95% CI: [48.26%, 48.69%])
   number: 31.19%
   other: 42.18%
   yes/no: 62.82%
   Est. GFLOPs/sample: 64.9

📊 TTT: K=1, rotation
   Accuracy: 47.19% (95% CI: [46.97%, 47.41%])
   number: 30.70%
   other: 41.22%
   yes/no: 60.79%
   Est. GFLOPs/sample: 64.9

📊 TTT: K=3, masked_patch
   Accuracy: 46.66% (95% CI: [46.44%, 46.87%])
   number: 28.83%
   other: 39.90%
   yes/no:

Generated figures land in `figures/` in the shared project folder. Open them locally.


## 9. Local CPU: Gradio Demo

Launch the interactive demo on your Mac. CPU inference is ~5–10s per query but fine for inspection. The cell **does not return** — Gradio runs a server until you interrupt the kernel.


In [6]:
# === LOCAL CPU CELL ===
!python demo/app.py \
    --config config/config.yaml \
    --checkpoint checkpoints/base/best.pt \
    --gate-checkpoint checkpoints/gate/best.pt \
    --port 7860


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[vqa] model loaded from checkpoints/base/best.pt
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Traceback (most recent call last):
  File "/Users/yugesh/Library/CloudStorage/GoogleDrive-yugeshreddysappidi@gmail.com/My Drive/AdaTTT/demo/app.py", line 553, in <module>
    main()
  File "/Users/yugesh/Library/CloudStorage/GoogleDrive-yugeshreddysappidi@gmail.com/My Drive/AdaTTT/demo/app.py", line 503, in main
    _TASKS["memotion2"] = _load_task_context(
                   

## 10. Memotion2 Cross-Task Evaluation

Run this section only after the VQA-v2 ablation and latency jobs you still care about are done. The cells below keep the VQA-v2 checkpoint safe while adding the Memotion2 experiment.


In [11]:
# === COLAB GPU CELL — Download Memotion via Kaggle ===
import os, json, shutil, glob, csv

memo_dir = "data/memotion2"
img_dir = os.path.join(memo_dir, "images")
os.makedirs(img_dir, exist_ok=True)

if os.path.exists(os.path.join(memo_dir, "train.json")):
    print("✅ Already prepared")
else:
    # Download via kagglehub (pre-installed on Colab)
    import kagglehub
    path = kagglehub.dataset_download("williamscott701/memotion-dataset-7k")
    print(f"Downloaded to: {path}")
    
    # Show contents
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        if os.path.isdir(full):
            print(f"  [DIR]  {item} ({len(os.listdir(full))} items)")
        else:
            print(f"  [FILE] {item} ({os.path.getsize(full)//1024}KB)")

    # Copy all images
    print("\nCopying images...")
    img_count = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".gif")):
                src = os.path.join(root, f)
                dst = os.path.join(img_dir, f)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    img_count += 1
    print(f"Copied {img_count} images")

    # Find and convert CSVs
    csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
    print(f"\nCSVs: {[os.path.basename(c) for c in csv_files]}")

    all_annotations = []
    for csv_path in csv_files:
        with open(csv_path, "r", errors="replace") as f:
            reader = csv.DictReader(f)
            print(f"  {os.path.basename(csv_path)}: {reader.fieldnames}")
            for i, row in enumerate(reader):
                img_name = (row.get("image_name") or row.get("Image_name") or 
                           row.get("img") or f"img_{i}.jpg")
                sentiment = (row.get("overall_sentiment") or row.get("sentiment") or 
                            row.get("label") or "neutral")
                text = (row.get("text_ocr") or row.get("text") or 
                       row.get("ocr") or "")
                all_annotations.append({
                    "id": f"memo_{len(all_annotations)}",
                    "image": str(img_name).strip(),
                    "text": str(text).strip(),
                    "sentiment": str(sentiment).lower().strip(),
                })

    if not all_annotations:
        raise RuntimeError("No annotations found!")

    import random
    random.seed(42)
    random.shuffle(all_annotations)
    split_idx = int(len(all_annotations) * 0.8)

    with open(os.path.join(memo_dir, "train.json"), "w") as f:
        json.dump(all_annotations[:split_idx], f)
    with open(os.path.join(memo_dir, "val.json"), "w") as f:
        json.dump(all_annotations[split_idx:], f)

    print(f"\n✅ Done! Train: {split_idx}, Val: {len(all_annotations)-split_idx}, Images: {len(os.listdir(img_dir))}")


100%|██████████| 695M/695M [00:02<00:00, 252MB/s]  

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/williamscott701/memotion-dataset-7k/versions/1
  [DIR]  memotion_dataset_7k (7 items)

Copying images...
Copied 6989 images

CSVs: ['reference.csv', 'labels.csv']
  reference.csv: ['', 'original_name', 'image_url', 'image_name']
  labels.csv: ['', 'image_name', 'text_ocr', 'text_corrected', 'humour', 'sarcasm', 'offensive', 'motivational', 'overall_sentiment']

✅ Done! Train: 11187, Val: 2797, Images: 6989


In [15]:
# === COLAB GPU CELL — Clean corrupt images + retrain ===
import os
from PIL import Image

img_dir = "data/memotion2/images"
bad = []
for f in os.listdir(img_dir):
    try:
        img = Image.open(os.path.join(img_dir, f))
        img.load()  # force full read
    except Exception:
        bad.append(f)
        os.remove(os.path.join(img_dir, f))

print(f"Removed {len(bad)} corrupt images")

# Retrain with strict_images=False as safety net
import shutil

vqa_ckpt = 'checkpoints/base/best.pt'
vqa_backup = 'checkpoints/base/best_vqa.pt'

# Temporarily set strict_images=False in config
import yaml
with open('config/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
orig_strict = cfg.get('strict_images', True)
cfg['strict_images'] = False
with open('config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

!python gpu/train_base.py \
    --config config/config.yaml \
    --dataset memotion2 \
    --epochs 15

# Restore strict_images
cfg['strict_images'] = orig_strict
with open('config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Move checkpoints
if os.path.exists(vqa_ckpt):
    shutil.move(vqa_ckpt, 'checkpoints/memotion2/best.pt')
    print('OK: Memotion2 checkpoint saved')
if os.path.exists(vqa_backup):
    shutil.copy2(vqa_backup, vqa_ckpt)
    print('OK: VQA checkpoint restored')


Removed 1 corrupt images
[11:35:44] INFO - Device: cuda, AMP: True
[11:35:44] INFO - Dataset: memotion2
[11:35:44] INFO - Epochs: 15, Batch size: 64, LR: 0.0001
Loading weights: 100% 198/198 [00:00<00:00, 927.66it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 1050.61it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: bert-base-uncased
Key                

In [16]:
# === COLAB GPU CELL ===
!python gpu/run_inference.py \
    --config config/config.yaml \
    --base-checkpoint checkpoints/memotion2/best.pt \
    --threshold 0.8 \
    --k 1 \
    --objective masked_patch \
    --dataset memotion2


[11:49:59] INFO - Adaptive Inference: τ=0.8, K=1, masked_patch, dataset=memotion2
[11:49:59] INFO - Device: cuda, AMP: True
Loading weights: 100% 198/198 [00:00<00:00, 949.72it/s, Materializing param=layernorm.weight]                                 
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100% 199/199 [00:00<00:00, 1116.25it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     